In [15]:
# ============================================
# PHASE 2 — STRICT DATA PROTOCOL (WORLD CLASS)
# ============================================

import pandas as pd
import numpy as np
import random
import os
import re

from sklearn.model_selection import GroupShuffleSplit


SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print("Phase 2 environment ready")

Phase 2 environment ready


In [2]:
#Cell 2

df = pd.read_csv("/Users/amarachi/Documents/AQA/Exp_3/regulqa_ambig_v11.csv")

print("Dataset loaded")
print("Rows:", len(df))
print("Columns:")
print(df.columns.tolist())

df.head()

Dataset loaded
Rows: 3314
Columns:
['id', 'source', 'tier', 'sector', 'document', 'req_text', 'ambig_presence', 'ambig_type', 'reg_clause', 'severity', 'notes']


,id,source,tier,sector,document,req_text,ambig_presence,ambig_type,reg_clause,severity,notes
0,PURE_000000,PURE,T1,automotive,1999 - dii.csv,DRAFT SOFTWARE REQUIREMENTS SPECIFICATION (SRS...,ambiguous,lexical;syntactic,ISO 26262-8 §6.4.3; ISO 29148 §5.2 (clarity & ...,high,"vague_term, comparative, modal_vague, passive,..."
1,PURE_000001,PURE,T1,automotive,1999 - dii.csv,REQUIREMENTS REQUIRED STATES AND MODES Environ...,ambiguous,lexical;syntactic,ISO 26262-8 §6.4.3; ISO 29148 §5.2 (clarity & ...,high,"vague_term, comparative, modal_vague, passive,..."
2,PURE_000002,PURE,T1,rail,1999 - dii.csv,REQUIRED STATES AND MODES Environmental Mode: ...,ambiguous,lexical;syntactic,ISO 26262-8 §6.4.3; ISO 29148 §5.2 (clarity & ...,high,"vague_term, modal_vague, passive"
3,PURE_000003,PURE,T1,rail,1999 - dii.csv,Environmental Mode: This is the real-time oper...,ambiguous,lexical;syntactic,ISO 26262-8 §6.4.3; ISO 29148 §5.2 (clarity & ...,high,"vague_term, modal_vague, passive"
4,PURE_000004,PURE,T1,general,1999 - dii.csv,COE Compliance. XML Services shall be segmente...,ambiguous,syntactic,ISO 29148 §5.2 (clarity & testability),low,passive


In [3]:
print("Columns in dataset:")
print(df.columns.tolist())

Columns in dataset:
['id', 'source', 'tier', 'sector', 'document', 'req_text', 'ambig_presence', 'ambig_type', 'reg_clause', 'severity', 'notes']


In [3]:
#Cell 3

# possible names for required columns
text_candidates = ["text","req_text","requirement","sentence","requirement_text"]
label_candidates = ["label","ambig_presence","ambiguity","is_ambiguous","class"]
source_candidates = ["source_document","document","doc","source","doc_id"]

def detect_column(candidates, columns):
    for c in candidates:
        if c in columns:
            return c
    return None

text_col = detect_column(text_candidates, df.columns)
label_col = detect_column(label_candidates, df.columns)
source_col = detect_column(source_candidates, df.columns)

print("Detected columns:")
print("text column:", text_col)
print("label column:", label_col)
print("source column:", source_col)

if text_col is None or label_col is None:
    raise ValueError("Could not detect text or label column automatically.")

Detected columns:
text column: req_text
label column: ambig_presence
source column: document


In [23]:
#if "is_synthetic" not in df.columns:
    #print("WARNING: 'is_synthetic' column not found.")

In [4]:
#Cell 4  Convert ambiguous/non-ambiguous to binary

df = df.rename(columns={
    text_col: "text",
    label_col: "label",
    source_col: "source_document"
})

if "source_document" not in df.columns:
    df["source_document"] = np.arange(len(df))

df = df[["text","label","source_document"]]

print("Standardized columns ready")
df.head()

Standardized columns ready


,text,label,source_document
0,DRAFT SOFTWARE REQUIREMENTS SPECIFICATION (SRS...,ambiguous,1999 - dii.csv
1,REQUIREMENTS REQUIRED STATES AND MODES Environ...,ambiguous,1999 - dii.csv
2,REQUIRED STATES AND MODES Environmental Mode: ...,ambiguous,1999 - dii.csv
3,Environmental Mode: This is the real-time oper...,ambiguous,1999 - dii.csv
4,COE Compliance. XML Services shall be segmente...,ambiguous,1999 - dii.csv


In [5]:
#Cell
import re

In [5]:
#Cell 5

def clean_text(x):

    x = str(x)

    x = x.lower()               # normalize case
    x = x.replace("\n"," ")     # remove newline artifacts
    x = re.sub(r"\s+"," ",x)    # collapse multiple spaces

    return x.strip()

df["text"] = df["text"].apply(clean_text)

print("Text normalization complete")

Text normalization complete


In [6]:
#Cell 6
df["label"] = df["label"].astype(str).str.lower()

df["label"] = df["label"].replace({
    "ambiguous":1,
    "not ambiguous":0,
    "non ambiguous":0,
    "non-ambiguous":0,
    "clear":0
})

df["label"] = pd.to_numeric(df["label"], errors="coerce")

df = df.dropna(subset=["label"])

df["label"] = df["label"].astype(int)

print("Label encoding complete")
print(df["label"].value_counts())

Label encoding complete
label
1    2458
0     856
Name: count, dtype: int64


/var/folders/nm/pwp8rhgj4z7fq1b63x3htxs00000gn/T/ipykernel_6488/1489537619.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["label"] = df["label"].replace({


In [7]:
#Cell 7
df = df[df["text"].str.len() > 5]

print("Dataset size after cleaning:", len(df))

Dataset size after cleaning: 3314


In [8]:
#Cell 8

before = len(df)

df = df.drop_duplicates(subset=["text"])

after = len(df)

print("Duplicates removed:", before - after)
print("Remaining samples:", after)

Duplicates removed: 0
Remaining samples: 3314


In [9]:
#Cell 9
print("Label distribution:")
print(df["label"].value_counts())
print(df["label"].value_counts(normalize=True))

Label distribution:
label
1    2458
0     856
Name: count, dtype: int64
label
1    0.741702
0    0.258298
Name: proportion, dtype: float64


In [10]:

#Cell 10 
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)

train_val_idx, test_idx = next(
    gss.split(df, groups=df["source_document"])
)

train_val = df.iloc[train_val_idx].reset_index(drop=True)
test = df.iloc[test_idx].reset_index(drop=True)

print("Train+Validation:", len(train_val))
print("Test:", len(test))

Train+Validation: 2959
Test: 355


In [11]:
#Cell 11
gss2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=SEED
)

train_idx, val_idx = next(
    gss2.split(train_val, groups=train_val["source_document"])
)

train = train_val.iloc[train_idx].reset_index(drop=True)
val = train_val.iloc[val_idx].reset_index(drop=True)

print("Train:", len(train))
print("Validation:", len(val))
print("Test:", len(test))

Train: 2338
Validation: 621
Test: 355


In [12]:
#cell 12 
train_docs = set(train["source_document"])
val_docs = set(val["source_document"])
test_docs = set(test["source_document"])

assert len(train_docs & val_docs) == 0
assert len(train_docs & test_docs) == 0
assert len(val_docs & test_docs) == 0

print("No document leakage detected")

No document leakage detected


In [13]:
#cell 13
os.makedirs("strict_splits", exist_ok=True)

train.to_csv("strict_splits/train.csv", index=False)
val.to_csv("strict_splits/val.csv", index=False)
test.to_csv("strict_splits/test.csv", index=False)

print("Strict splits saved")

Strict splits saved
